# Pandas Revision — The `mpg` Dataset

398 cars sold in the United States between 1970 and 1982, with fuel economy, engine specs,
weight and country of origin. It is small enough to check by eye and messy enough to be real.

Eighteen questions. They run roughly in the order we covered the topics, so if you are stuck
on Q11 the answer is not in Q17 — it is in your notes from the `idxmax` section.

Two of these questions (**Q10** and **Q17**) are not about syntax. You will get a number in
about thirty seconds, and then the actual work begins: deciding whether that number means
what it appears to mean. Those are the ones worth your time.

**The columns**

| Column | Meaning |
|--------|---------|
| `mpg` | Miles per gallon — higher is more efficient |
| `cylinders` | Engine cylinders (3, 4, 5, 6 or 8) |
| `displacement` | Engine size, cubic inches |
| `horsepower` | Engine power |
| `weight` | Kerb weight, pounds |
| `acceleration` | Seconds from 0 to 60 mph — *lower* is faster |
| `model_year` | Two-digit year: 70 = 1970 |
| `origin` | `usa`, `japan` or `europe` |
| `name` | Make and model |

In [1]:
import pandas as pd

cars = pd.read_csv('../../Data/mpg.csv')
cars.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


---
## Q1. Inspect the data

Before touching anything, find out what you are holding.

1. How many rows and columns?
2. What is the dtype of each column?
3. Which columns have missing values, and how many?

In [2]:
print(cars.shape)
print()
print(cars.dtypes)
print()
print(cars.isna().sum())

(398, 9)

mpg             float64
cylinders         int64
displacement    float64
horsepower      float64
weight            int64
acceleration    float64
model_year        int64
origin           object
name             object
dtype: object

mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64


**Answer.** 398 rows and 9 columns. `name` and `origin` are text (object), `cylinders`, `weight` and `model_year` are int, the rest are float. Only `horsepower` has missing values — 6 of them.

---
## Q2. Select columns

Show only `name`, `mpg` and `origin`, for the first 5 rows.

In [3]:
cars[['name', 'mpg', 'origin']].head()

,name,mpg,origin
0,chevrolet chevelle malibu,18.0,usa
1,buick skylark 320,15.0,usa
2,plymouth satellite,18.0,usa
3,amc rebel sst,16.0,usa
4,ford torino,17.0,usa


---
## Q3. Filter rows

How many cars manage more than 30 mpg? Of those, how many are Japanese?

In [4]:
above_30 = cars[cars['mpg'] > 30]
print("cars above 30 mpg:", len(above_30))

japanese = above_30[above_30['origin'] == 'japan']
print("of those, japanese:", len(japanese))

cars above 30 mpg: 85
of those, japanese: 46


**Answer.** 85 cars do more than 30 mpg, and 46 of them are Japanese — more than half.

---
## Q4. Handle missing values

`horsepower` has blanks.

1. Which origins do the missing rows belong to?
2. Fill the blanks with the median horsepower.
3. Confirm there are no blanks left.

In [5]:
# 1. which origins have the blanks
print(cars[cars['horsepower'].isna()]['origin'].value_counts())

# 2. fill with median
median_hp = cars['horsepower'].median()
cars['horsepower'] = cars['horsepower'].fillna(median_hp)

# 3. check
print("blanks left:", cars['horsepower'].isna().sum())

origin
usa       4
europe    2
Name: count, dtype: int64
blanks left: 0


**Answer.** 4 of the missing rows are American cars and 2 are European.

---
## Q5. Create a new column

Power-to-weight ratio is a better predictor of how a car *feels* than horsepower alone.

Add a column `power_to_weight` = horsepower ÷ weight. Then show the 5 cars with the highest ratio.

In [6]:
cars['power_to_weight'] = cars['horsepower'] / cars['weight']

cars.sort_values('power_to_weight', ascending=False)[['name', 'horsepower', 'weight', 'power_to_weight']].head()

,name,horsepower,weight,power_to_weight
13,buick estate wagon (sw),225.0,3086,0.072910
116,pontiac grand prix,230.0,4278,0.053763
330,renault lecar deluxe,93.5,1835,0.050954
8,pontiac catalina,225.0,4425,0.050847
23,bmw 2002,113.0,2234,0.050582


---
## Q6. Count with value_counts

1. How many cars come from each origin?
2. How many cars have each cylinder count?
3. Show the origin breakdown as percentages instead of counts.

In [7]:
print(cars['origin'].value_counts())
print()
print(cars['cylinders'].value_counts())
print()
print((cars['origin'].value_counts(normalize=True) * 100).round(1))

origin
usa       249
japan      79
europe     70
Name: count, dtype: int64

cylinders
4    204
8    103
6     84
3      4
5      3
Name: count, dtype: int64

origin
usa       62.6
japan     19.8
europe    17.6
Name: proportion, dtype: float64


---
## Q7. Named aggregation

Build one table with a row per origin and these columns:
`cars`, `avg_mpg`, `avg_weight`, `avg_cylinders`. Round to 1 decimal.

In [8]:
summary = cars.groupby('origin').agg(
    cars=('mpg', 'size'),
    avg_mpg=('mpg', 'mean'),
    avg_weight=('weight', 'mean'),
    avg_cylinders=('cylinders', 'mean')
).round(1)

summary

,cars,avg_mpg,avg_weight,avg_cylinders
origin,,,,
europe,70,27.9,2423.3,4.2
japan,79,30.5,2221.2,4.1
usa,249,20.1,3361.9,6.2


---
## Q8. size vs count

For each origin, show how many cars there are **and** how many have a horsepower recorded.

*Run this on a fresh copy of the data — if you filled the blanks in Q4, they are gone.*

In [9]:
fresh = pd.read_csv('../../Data/mpg.csv')

fresh.groupby('origin')['horsepower'].agg(['size', 'count'])
# size counts every row, count skips the NaN values

,size,count
origin,,
europe,70,68
japan,79,79
usa,249,245


---
## Q9. Group by two columns

Average mpg for every combination of `origin` and `cylinders`. Then reshape it with
`.unstack()` so origins are rows and cylinder counts are columns.

In [10]:
by_two = cars.groupby(['origin', 'cylinders'])['mpg'].mean().round(1)
print(by_two)

origin  cylinders
europe  4            28.4
        5            27.4
        6            20.1
japan   3            20.6
        4            31.6
        6            23.9
usa     4            27.8
        6            19.7
        8            15.0
Name: mpg, dtype: float64


---
## Q10. ⭐ The confound

Someone shows you this:

> *Japanese cars average 30.5 mpg. American cars average 20.1. Japanese engineering is
> simply better at fuel efficiency.*

Both numbers are correct — you computed them yourself in Q7.

**Is the conclusion correct?** Investigate. Compare like with like, and state clearly what
the data does and does not support.

In [11]:
by_two.unstack()

cylinders,3,4,5,6,8
origin,,,,,
europe,NaN,28.4,27.4,20.1,NaN
japan,20.6,31.6,NaN,23.9,NaN
usa,NaN,27.8,NaN,19.7,15.0


In [12]:
# compare like with like: only 4 cylinder cars
four = cars[cars['cylinders'] == 4]
four.groupby('origin', observed=True)[['mpg', 'weight']].mean().round(1)

,mpg,weight
origin,,
europe,28.4,2330.0
japan,31.6,2153.5
usa,27.8,2437.2


In [13]:
# and cars of similar weight
cars['weight_band'] = pd.cut(cars['weight'], [1500, 2000, 2500, 3000, 3500, 5500])
cars.groupby(['weight_band', 'origin'], observed=True)['mpg'].mean().round(1).unstack()

origin,europe,japan,usa
weight_band,,,
"(1500, 2000]",31.2,34.4,34.2
"(2000, 2500]",30.4,30.1,28.4
"(2500, 3000]",22.8,26.0,24.3
"(3000, 3500]",23.5,NaN,19.4
"(3500, 5500]",21.0,NaN,15.1


**Answer.** No, the conclusion does not hold up the way it is stated.

- The two averages compare very different groups of cars. Most American cars in the data are big 6 and 8 cylinder cars (avg weight about 3360 lb), while Japanese cars are almost all small 4 cylinder cars (about 2220 lb). Heavy cars with big engines use more fuel, whatever country builds them.
- If we only look at 4 cylinder cars, the gap drops from about 10 mpg to about 4 mpg (Japan 31.6, USA 27.8, Europe 28.4).
- Within the same weight band the numbers are even closer, for example 2000–2500 lb: Europe 30.4, Japan 30.1, USA 28.4.

So what the data supports: Japan sold smaller and lighter cars, and that explains most of the difference. Among similar cars Japan is a little ahead, but "Japanese engineering is simply better" is too strong — most of the 10 mpg gap is about **what kind of cars** each country made, not how well they made them.

---
## Q11. idxmax

Which specific car has the best fuel economy in each origin? Show the car's name and mpg,
not just the number.

In [14]:
best = cars.groupby('origin')['mpg'].idxmax()
cars.loc[best, ['origin', 'name', 'mpg']]

,origin,name,mpg
325,europe,vw rabbit c (diesel),44.3
322,japan,mazda glc,46.6
344,usa,plymouth champ,39.0


---
## Q12. astype and to_csv

1. Convert `origin` to the `category` dtype and report the memory saved.
2. Save your Q7 summary table to `origin_summary.csv` — with the origin column intact.
3. Read the file back and confirm nothing was lost.

In [15]:
# 1.
before = cars['origin'].memory_usage(deep=True)
cars['origin'] = cars['origin'].astype('category')
after = cars['origin'].memory_usage(deep=True)
print("before:", before, "bytes")
print("after:", after, "bytes")
print("saved:", before - after, "bytes")

# 2. reset_index so origin is saved as a normal column
summary.reset_index().to_csv('origin_summary.csv', index=False)

# 3.
check = pd.read_csv('origin_summary.csv')
print(check)
print(check.equals(summary.reset_index()))

before: 21196 bytes
after: 799 bytes
saved: 20397 bytes
   origin  cars  avg_mpg  avg_weight  avg_cylinders
0  europe    70     27.9      2423.3            4.2
1   japan    79     30.5      2221.2            4.1
2     usa   249     20.1      3361.9            6.2
True


---
## Q13. Sorting

Show the 10 heaviest cars, with name, weight and mpg. What do you notice about their mpg?

In [16]:
cars.sort_values('weight', ascending=False)[['name', 'weight', 'mpg']].head(10)

,name,weight,mpg
44,pontiac safari (sw),5140,13.0
103,chevrolet impala,4997,11.0
42,dodge monaco (sw),4955,12.0
90,mercury marquis brougham,4952,12.0
95,buick electra 225 custom,4951,12.0
104,ford country,4906,12.0
43,ford country squire (sw),4746,13.0
94,chrysler new yorker brougham,4735,13.0
28,hi 1200d,4732,9.0
137,buick century luxus (sw),4699,13.0


**Answer.** All ten heaviest cars are big American cars and their mpg is very low, between 9 and 13. Weight and fuel economy clearly go in opposite directions.

---
## Q14. Compound filter

Find American cars that are 4-cylinder, from model year 1978 or later, and manage more than
25 mpg. How many are there? Show the 5 most efficient.

In [17]:
filtered = cars[(cars['origin'] == 'usa') &
                (cars['cylinders'] == 4) &
                (cars['model_year'] >= 78) &
                (cars['mpg'] > 25)]

print("how many:", len(filtered))
filtered.sort_values('mpg', ascending=False)[['name', 'model_year', 'mpg']].head()

how many: 37


,name,model_year,mpg
344,plymouth champ,81,39.0
378,plymouth horizon miser,82,38.0
245,ford fiesta,78,36.1
391,dodge charger 2.2,82,36.0
379,mercury lynx l,82,36.0


There are 37 such cars.

---
## Q15. String methods

The `name` column holds make and model together, like `chevrolet chevelle malibu`.

1. Extract the brand — the first word — into a new `brand` column.
2. List every distinct brand.
3. Something is wrong with that list. Find the problem and fix it.

In [18]:
# 1.
cars['brand'] = cars['name'].str.split().str[0]

# 2.
print(sorted(cars['brand'].unique()))
print(cars['brand'].nunique(), "brands")

['amc', 'audi', 'bmw', 'buick', 'cadillac', 'capri', 'chevroelt', 'chevrolet', 'chevy', 'chrysler', 'datsun', 'dodge', 'fiat', 'ford', 'hi', 'honda', 'maxda', 'mazda', 'mercedes', 'mercedes-benz', 'mercury', 'nissan', 'oldsmobile', 'opel', 'peugeot', 'plymouth', 'pontiac', 'renault', 'saab', 'subaru', 'toyota', 'toyouta', 'triumph', 'vokswagen', 'volkswagen', 'volvo', 'vw']
37 brands


In [19]:
# 3. same brands are written in different ways / with spelling mistakes
#    chevy, chevroelt -> chevrolet    vw, vokswagen -> volkswagen
#    toyouta -> toyota    maxda -> mazda    mercedes-benz -> mercedes
fix = {
    'chevy': 'chevrolet',
    'chevroelt': 'chevrolet',
    'vw': 'volkswagen',
    'vokswagen': 'volkswagen',
    'toyouta': 'toyota',
    'maxda': 'mazda',
    'mercedes-benz': 'mercedes'
}
cars['brand'] = cars['brand'].replace(fix)

print(sorted(cars['brand'].unique()))
print(cars['brand'].nunique(), "brands")

['amc', 'audi', 'bmw', 'buick', 'cadillac', 'capri', 'chevrolet', 'chrysler', 'datsun', 'dodge', 'fiat', 'ford', 'hi', 'honda', 'mazda', 'mercedes', 'mercury', 'nissan', 'oldsmobile', 'opel', 'peugeot', 'plymouth', 'pontiac', 'renault', 'saab', 'subaru', 'toyota', 'triumph', 'volkswagen', 'volvo']
30 brands


**Answer.** The problem is that the same brand is written in more than one way — short names (`chevy`, `vw`), typos (`chevroelt`, `vokswagen`, `toyouta`, `maxda`) and `mercedes` vs `mercedes-benz`. Without fixing it, a brand gets split into several groups and every count/average by brand is wrong. After mapping them, 37 names become 30 real brands.

---
## Q16. Brand summary

Using your cleaned `brand` column: for every brand with at least 15 cars, show the number of
cars and the average mpg, sorted from most to least efficient.

In [20]:
brand_summary = cars.groupby('brand').agg(
    cars=('mpg', 'size'),
    avg_mpg=('mpg', 'mean')
).round(1)

brand_summary[brand_summary['cars'] >= 15].sort_values('avg_mpg', ascending=False)

,cars,avg_mpg
brand,,
volkswagen,22,31.8
datsun,23,31.1
toyota,26,28.2
dodge,28,22.1
plymouth,31,21.7
chevrolet,47,20.2
pontiac,16,20.0
ford,51,19.7
buick,17,19.2


---
## Q17. ⭐ Change over time

1. What was the average mpg of the 1970 fleet? The 1982 fleet?
2. Someone concludes: *"Japanese cars got much more efficient over the decade — that is why
   they took the market."* Check it. Did Japanese cars improve more than everyone else's?

In [21]:
year_avg = cars.groupby('model_year')['mpg'].mean().round(2)
print("1970 fleet:", year_avg[70])
print("1982 fleet:", year_avg[82])

1970 fleet: 17.69
1982 fleet: 31.71


In [22]:
by_year = cars.groupby(['model_year', 'origin'], observed=True)['mpg'].mean().unstack().round(1)
counts = cars.groupby(['model_year', 'origin'], observed=True)['mpg'].size().unstack()

print(by_year.loc[[70, 82]])
print()
print("change 1970 -> 1982:")
print((by_year.loc[82] - by_year.loc[70]).round(1))
print()
print("number of cars in those years:")
print(counts.loc[[70, 82]])

origin      europe  japan   usa
model_year                     
70            25.2   25.5  15.3
82            40.0   34.9  29.4

change 1970 -> 1982:
origin
europe    14.8
japan      9.4
usa       14.1
dtype: float64

number of cars in those years:
origin      europe  japan  usa
model_year                    
70               5      2   22
82               2      9   20


**Answer.**

1. The average car went from about **17.7 mpg in 1970** to **31.7 mpg in 1982** — the whole fleet got a lot better.
2. The claim does not hold. Japanese cars went from 25.5 to 34.9 mpg (about **+9**), but American cars went from 15.3 to 29.4 (about **+14**) and European cars from 25.2 to 40.0 (about **+15**). Japanese cars improved the *least* of the three.

Japan was already efficient in 1970 — the advantage was there from the start, it did not come from improving faster. Also be careful: 1970 has only 2 Japanese cars and 1982 only 2 European cars, so those single year averages are based on very few cars.

---
## Q18. idxmin

For each cylinder count, which car has the *worst* fuel economy?

In [23]:
worst = cars.groupby('cylinders')['mpg'].idxmin()
cars.loc[worst, ['cylinders', 'name', 'mpg']]

,cylinders,name,mpg
111,3,maxda rx3,18.0
76,4,volvo 145e (sw),18.0
274,5,audi 5000,20.3
128,6,chevrolet nova,15.0
28,8,hi 1200d,9.0


---
## When you are done

Q10 and Q17 are the two that matter. Anyone can look up `.agg()` later; recognising that an
average is comparing two different populations is the skill that does not come back on its own.

If you finished early, one more: **is `acceleration` correlated with `mpg`?** Careful — lower
acceleration numbers mean faster cars.

In [24]:
print(cars['acceleration'].corr(cars['mpg']).round(2))
print(cars['acceleration'].corr(cars['horsepower']).round(2))

0.42
-0.69


**Bonus answer.** Yes, there is a moderate positive correlation (about 0.42). But higher acceleration number means a *slower* car, so it actually says: slower cars tend to have better mpg, and fast powerful cars use more fuel. The -0.69 with horsepower confirms that faster cars (low number) have more horsepower.